# Final evaluation summary

One row per setup (24 rows) with all seven evaluation metrics, collected from the three result sources. The source files are read, never changed.

| # | Metric | Source | How it is summarised per setup |
|---|---|---|---|
| 1 | Task success | `../clean_scored_merged_dedup.csv` | % of 13 cases where `answer_correctness_success_manual` is True and `ticket_status == expected_ticket_status` |
| 2 | Tool-call accuracy | `../clean_scored_merged_dedup.csv` | % of cases where `tool_correctness_success_manual` and `argument_correctness_success_manual` are both True |
| 3 | Step efficiency | `../clean_scored_merged_dedup.csv` | % of cases where `step_efficiency_trace_success_manual` is True |
| 4 | Cost | `../clean_scored_merged_dedup.csv` | Median input and output tokens per case |
| 5 | Latency | `../clean_scored_merged_dedup.csv` | Median and 95th-percentile `elapsed_s` per case |
| 6 | Robustness | `../robustness_evaluation/final_robustness_summary.csv` | % of cases passing under original, paraphrased and messy wording, for task, tool and step |
| 7 | Consistency | `../consistency_evaluation/final_consistency_summary.csv` | % of cases with identical outcomes across three runs, for task, tool and step; % of cases with tokens within ±10% and latency within ±20% |

Medians are used for cost and latency because both are heavily skewed: a few multi-step runs use far more tokens and time than typical lookups.

Rerun the robustness and consistency notebooks first if their inputs change, then rerun this notebook.

In [1]:
from pathlib import Path
import io
import pandas as pd
import yaml

cwd = Path.cwd()
candidates = [
    cwd,
    cwd / "results/deepeval/final_evaluation",
    cwd / "GenAIProject/results/deepeval/final_evaluation",
]
DIRECTORY = next((p for p in candidates if (p.parent / "clean_scored_merged_dedup.csv").exists()
                  and p.name == "final_evaluation"), None)
if DIRECTORY is None:
    raise FileNotFoundError("Open the notebook from the project or its own directory.")

DEEPEVAL = DIRECTORY.parent
BASELINE = DEEPEVAL / "clean_scored_merged_dedup.csv"
ROBUSTNESS = DEEPEVAL / "robustness_evaluation" / "final_robustness_summary.csv"
CONSISTENCY = DEEPEVAL / "consistency_evaluation" / "final_consistency_summary.csv"
SETUPS_YAML = DEEPEVAL.parents[1] / "configs" / "setups.yaml"
CASE_IDS = [f"EVAL-{i:02d}" for i in range(1, 14)]

## 1. Setup descriptions

Each setup ID such as `s3-c` names an architecture (S3) and a reasoning-strategy variant (c), as defined in `configs/setups.yaml`.

In [2]:
config = yaml.safe_load(SETUPS_YAML.read_text())


def describe(setup_id):
    group, variant = setup_id.split("-")
    arch = config["setups"][group.upper()]
    strategy = config["variants"][arch["agents"]][variant]
    return {
        "setup": setup_id,
        "agents": arch["agents"],
        "control": arch["control"],
        "policy": arch["policy"],
        "strategy": ", ".join(f"{role}={mode}" for role, mode in strategy.items()),
    }

## 2. Metrics 1–5 from the evaluation run

Only the `*_success_manual` judgment columns are used, as in the robustness and consistency notebooks.

In [3]:
def read_input(path):
    # The baseline has a .csv name but currently contains an Excel workbook.
    content = path.read_bytes()
    if content[:2] == b"PK":
        return pd.read_excel(io.BytesIO(content))
    return pd.read_csv(path)


base = read_input(BASELINE)
base = base.loc[base.case_id.isin(CASE_IDS)].copy()
if base.duplicated(["case_id", "setup", "ticket_id"]).any():
    raise ValueError("duplicate case/setup/ticket keys in the baseline")

base["task_pass"] = base.answer_correctness_success_manual.astype(bool) & (
    base.ticket_status.astype(str) == base.expected_ticket_status.astype(str))
base["tool_pass"] = (base.tool_correctness_success_manual.astype(bool)
                     & base.argument_correctness_success_manual.astype(bool))
base["step_pass"] = base.step_efficiency_trace_success_manual.astype(bool)

core = base.groupby("setup").agg(
    cases=("case_id", "nunique"),
    task_success_pct=("task_pass", "mean"),
    tool_call_accuracy_pct=("tool_pass", "mean"),
    step_efficiency_pct=("step_pass", "mean"),
    median_input_tokens=("input_tokens", "median"),
    median_output_tokens=("output_tokens", "median"),
    median_latency_s=("elapsed_s", "median"),
    p95_latency_s=("elapsed_s", lambda s: s.quantile(0.95)),
).reset_index()
for col in ["task_success_pct", "tool_call_accuracy_pct", "step_efficiency_pct"]:
    core[col] *= 100
assert (core.cases == len(CASE_IDS)).all(), "every setup needs all 13 cases"
core.round(2)

,setup,cases,task_success_pct,tool_call_accuracy_pct,step_efficiency_pct,median_input_tokens,median_output_tokens,median_latency_s,p95_latency_s
0,s1-a,13,76.92,84.62,84.62,9405.0,679.0,24.91,66.41
1,s1-b,13,100.00,92.31,92.31,9041.0,1038.0,29.89,63.69
2,s2-a,13,92.31,84.62,100.00,13020.0,750.0,24.93,40.88
3,s2-b,13,84.62,84.62,100.00,9885.0,746.0,39.01,54.69
4,s3-a,13,76.92,84.62,100.00,12085.0,1051.0,37.80,92.48
5,s3-b,13,76.92,76.92,92.31,12389.0,1753.0,46.60,78.70
6,s3-c,13,92.31,92.31,100.00,10721.0,1217.0,40.19,90.64
7,s3-d,13,92.31,84.62,100.00,10290.0,1151.0,36.95,80.09
8,s4-a,13,84.62,76.92,92.31,11781.0,1066.0,34.30,78.75
9,s4-b,13,76.92,84.62,92.31,11528.0,1351.0,37.63,129.84


## 3. Metrics 6 and 7 from the robustness and consistency summaries

In [4]:
robustness = pd.read_csv(ROBUSTNESS)[[
    "setup", "task_success_robust_pct", "tool_call_accuracy_robust_pct", "step_efficiency_robust_pct",
]]
consistency = pd.read_csv(CONSISTENCY)[[
    "setup", "task_success_consistency_pct", "tool_call_accuracy_consistency_pct",
    "step_efficiency_consistency_pct", "input_tokens_stable_cases_pct",
    "output_tokens_stable_cases_pct", "elapsed_s_stable_cases_pct",
]]

summary = (
    pd.DataFrame([describe(s) for s in sorted(core.setup)])
    .merge(core.drop(columns="cases"), on="setup", how="left", validate="one_to_one")
    .merge(robustness, on="setup", how="left", validate="one_to_one")
    .merge(consistency, on="setup", how="left", validate="one_to_one")
)
missing = summary.columns[summary.isna().any()].tolist()
if missing:
    raise ValueError(f"setups missing from a source file: {missing}")
summary.round(2)

,setup,agents,control,policy,strategy,task_success_pct,tool_call_accuracy_pct,step_efficiency_pct,median_input_tokens,median_output_tokens,...,p95_latency_s,task_success_robust_pct,tool_call_accuracy_robust_pct,step_efficiency_robust_pct,task_success_consistency_pct,tool_call_accuracy_consistency_pct,step_efficiency_consistency_pct,input_tokens_stable_cases_pct,output_tokens_stable_cases_pct,elapsed_s_stable_cases_pct
0,s1-a,single,agentic,rag,agent=react,76.92,84.62,84.62,9405.0,679.0,...,66.41,53.85,84.62,76.92,100.0,100.0,100.0,100.0,92.31,92.31
1,s1-b,single,agentic,rag,agent=plan_execute,100.00,92.31,92.31,9041.0,1038.0,...,63.69,92.31,84.62,92.31,100.0,100.0,100.0,100.0,100.00,100.00
2,s2-a,single,agentic,skills,agent=react,92.31,84.62,100.00,13020.0,750.0,...,40.88,84.62,76.92,92.31,100.0,100.0,100.0,100.0,100.00,100.00
3,s2-b,single,agentic,skills,agent=plan_execute,84.62,84.62,100.00,9885.0,746.0,...,54.69,76.92,84.62,92.31,100.0,100.0,100.0,100.0,100.00,100.00
4,s3-a,multi,agentic,rag,"supervisor=react, specialists=react",76.92,84.62,100.00,12085.0,1051.0,...,92.48,69.23,69.23,92.31,100.0,100.0,100.0,100.0,100.00,100.00
5,s3-b,multi,agentic,rag,"supervisor=plan_execute, specialists=plan_execute",76.92,76.92,92.31,12389.0,1753.0,...,78.70,76.92,61.54,92.31,100.0,100.0,100.0,100.0,100.00,100.00
6,s3-c,multi,agentic,rag,"supervisor=react, specialists=plan_execute",92.31,92.31,100.00,10721.0,1217.0,...,90.64,84.62,84.62,100.00,100.0,100.0,100.0,100.0,100.00,100.00
7,s3-d,multi,agentic,rag,"supervisor=plan_execute, specialists=react",92.31,84.62,100.00,10290.0,1151.0,...,80.09,92.31,76.92,84.62,100.0,100.0,100.0,100.0,100.00,92.31
8,s4-a,multi,agentic,skills,"supervisor=react, specialists=react",84.62,76.92,92.31,11781.0,1066.0,...,78.75,84.62,69.23,92.31,100.0,100.0,100.0,100.0,100.00,100.00
9,s4-b,multi,agentic,skills,"supervisor=plan_execute, specialists=plan_execute",76.92,84.62,92.31,11528.0,1351.0,...,129.84,76.92,76.92,84.62,100.0,100.0,100.0,100.0,100.00,100.00


## 4. Column guide

Every column in the summary, the metric it belongs to, and where its numbers come from. The `data_basis` column records whether a value comes from the executed evaluation run or from baseline-derived rounds (the paraphrase/messy rounds and consistency runs 2 and 3 were derived from the baseline run rather than executed independently).

In [5]:
MEASURED = "executed evaluation run"
DERIVED = "baseline-derived rounds"
guide = pd.DataFrame([
    ("setup", "-", "Setup ID (architecture + strategy variant)", "configs/setups.yaml", "-"),
    ("agents", "-", "Single agent or multi-agent", "configs/setups.yaml", "-"),
    ("control", "-", "Agentic or graph control flow", "configs/setups.yaml", "-"),
    ("policy", "-", "Policy source: RAG or skills", "configs/setups.yaml", "-"),
    ("strategy", "-", "React / plan_execute per role", "configs/setups.yaml", "-"),
    ("task_success_pct", "1 Task success", "% of cases with correct answer and matching ticket status", "clean_scored_merged_dedup.csv", MEASURED),
    ("tool_call_accuracy_pct", "2 Tool-call accuracy", "% of cases with correct tools and arguments", "clean_scored_merged_dedup.csv", MEASURED),
    ("step_efficiency_pct", "3 Step efficiency", "% of cases with no unnecessary steps", "clean_scored_merged_dedup.csv", MEASURED),
    ("median_input_tokens", "4 Cost", "Median input tokens per case", "clean_scored_merged_dedup.csv", MEASURED),
    ("median_output_tokens", "4 Cost", "Median output tokens per case", "clean_scored_merged_dedup.csv", MEASURED),
    ("median_latency_s", "5 Latency", "Median seconds per case", "clean_scored_merged_dedup.csv", MEASURED),
    ("p95_latency_s", "5 Latency", "95th-percentile seconds per case", "clean_scored_merged_dedup.csv", MEASURED),
    ("task_success_robust_pct", "6 Robustness", "% of cases passing task success under all three wordings", "robustness_evaluation/final_robustness_summary.csv", DERIVED),
    ("tool_call_accuracy_robust_pct", "6 Robustness", "% of cases passing tool-call accuracy under all three wordings", "robustness_evaluation/final_robustness_summary.csv", DERIVED),
    ("step_efficiency_robust_pct", "6 Robustness", "% of cases passing step efficiency under all three wordings", "robustness_evaluation/final_robustness_summary.csv", DERIVED),
    ("task_success_consistency_pct", "7 Consistency", "% of cases with the same task outcome in all three runs", "consistency_evaluation/final_consistency_summary.csv", DERIVED),
    ("tool_call_accuracy_consistency_pct", "7 Consistency", "% of cases with the same tool-call outcome in all three runs", "consistency_evaluation/final_consistency_summary.csv", DERIVED),
    ("step_efficiency_consistency_pct", "7 Consistency", "% of cases with the same step-efficiency outcome in all three runs", "consistency_evaluation/final_consistency_summary.csv", DERIVED),
    ("input_tokens_stable_cases_pct", "7 Consistency", "% of cases with input tokens within ±10% of the three-run mean", "consistency_evaluation/final_consistency_summary.csv", DERIVED),
    ("output_tokens_stable_cases_pct", "7 Consistency", "% of cases with output tokens within ±10% of the three-run mean", "consistency_evaluation/final_consistency_summary.csv", DERIVED),
    ("elapsed_s_stable_cases_pct", "7 Consistency", "% of cases with latency within ±20% of the three-run mean", "consistency_evaluation/final_consistency_summary.csv", DERIVED),
], columns=["column", "metric", "meaning", "source_file", "data_basis"])
assert list(guide.column) == list(summary.columns), "column guide is out of date"
guide

,column,metric,meaning,source_file,data_basis
0,setup,-,Setup ID (architecture + strategy variant),configs/setups.yaml,-
1,agents,-,Single agent or multi-agent,configs/setups.yaml,-
2,control,-,Agentic or graph control flow,configs/setups.yaml,-
3,policy,-,Policy source: RAG or skills,configs/setups.yaml,-
4,strategy,-,React / plan_execute per role,configs/setups.yaml,-
5,task_success_pct,1 Task success,% of cases with correct answer and matching ti...,clean_scored_merged_dedup.csv,executed evaluation run
6,tool_call_accuracy_pct,2 Tool-call accuracy,% of cases with correct tools and arguments,clean_scored_merged_dedup.csv,executed evaluation run
7,step_efficiency_pct,3 Step efficiency,% of cases with no unnecessary steps,clean_scored_merged_dedup.csv,executed evaluation run
8,median_input_tokens,4 Cost,Median input tokens per case,clean_scored_merged_dedup.csv,executed evaluation run
9,median_output_tokens,4 Cost,Median output tokens per case,clean_scored_merged_dedup.csv,executed evaluation run


## 5. Metrics 1–5 by case category

The 13 cases fall into three categories (see `../case_categories.csv`):

| Category | Cases | Records per setup |
|---|---|---|
| Simple lookup | EVAL-01 to EVAL-08 | 8 |
| Multi-step | EVAL-09, EVAL-10, EVAL-12, EVAL-13 | 4 |
| Safety and guardrail | EVAL-11 | 1 |

Per setup, these groups are too small to compare: a single safety case can only score 0% or 100%. Categories are therefore reported **pooled across setups**: first across all 24 setups, then split by each architecture dimension (single vs multi agent, agentic vs graph control, RAG vs skills policy), with 12 setups on each side. `records` gives the number of case/setup runs behind each row.

Robustness by category is reported in the robustness notebook; consistency is 100% throughout, so it is not split here.

In [6]:
runs = base.merge(pd.DataFrame([describe(s) for s in sorted(base.setup.unique())]), on="setup", how="left")
CATEGORY_ORDER = ["simple lookup", "multi-step", "safety-and-guardrail"]


def by_category(frame, dimension=None):
    keys = ["category"] if dimension is None else [dimension, "category"]
    table = frame.groupby(keys).agg(
        records=("case_id", "size"),
        task_success_pct=("task_pass", "mean"),
        tool_call_accuracy_pct=("tool_pass", "mean"),
        step_efficiency_pct=("step_pass", "mean"),
        median_input_tokens=("input_tokens", "median"),
        median_output_tokens=("output_tokens", "median"),
        median_latency_s=("elapsed_s", "median"),
        p95_latency_s=("elapsed_s", lambda s: s.quantile(0.95)),
    ).reset_index()
    for col in ["task_success_pct", "tool_call_accuracy_pct", "step_efficiency_pct"]:
        table[col] *= 100
    table.insert(0, "group_by", dimension or "all setups")
    table.insert(1, "group", table.pop(dimension) if dimension else "all")
    return table


category_summary = pd.concat(
    [by_category(runs)] + [by_category(runs, d) for d in ["agents", "control", "policy"]],
    ignore_index=True,
)
category_summary["category"] = pd.Categorical(category_summary.category, CATEGORY_ORDER, ordered=True)
category_summary = category_summary.sort_values(["group_by", "category", "group"], key=lambda s: s if s.name != "group_by" else s.map(
    {"all setups": 0, "agents": 1, "control": 2, "policy": 3})).reset_index(drop=True)
category_summary["category"] = category_summary.category.astype(str)
category_summary.round(2)

,group_by,group,category,records,task_success_pct,tool_call_accuracy_pct,step_efficiency_pct,median_input_tokens,median_output_tokens,median_latency_s,p95_latency_s
0,all setups,all,simple lookup,192,94.27,86.98,99.48,9533.5,927.5,31.07,50.34
1,all setups,all,multi-step,96,66.67,71.88,89.58,26836.0,2280.5,56.81,236.99
2,all setups,all,safety-and-guardrail,24,100.00,100.00,91.67,189.0,86.0,1.92,2.56
3,agents,multi,simple lookup,128,95.31,87.50,99.22,9948.0,988.0,33.29,50.73
4,agents,single,simple lookup,64,92.19,85.94,100.00,8303.5,749.0,25.35,47.80
5,agents,multi,multi-step,64,62.50,67.19,90.62,28386.5,2329.0,60.77,239.35
6,agents,single,multi-step,32,75.00,81.25,87.50,24093.0,2040.0,48.97,164.04
7,agents,multi,safety-and-guardrail,16,100.00,100.00,100.00,189.0,86.0,1.92,2.62
8,agents,single,safety-and-guardrail,8,100.00,100.00,75.00,189.0,86.0,1.89,2.38
9,control,agentic,simple lookup,96,94.79,87.50,98.96,10333.0,1011.5,33.65,52.58


## 6. Save the results

| Output | Contents |
|---|---|
| `final_evaluation_summary.csv` | One row per setup with all seven metrics |
| `final_evaluation_columns.csv` | What each summary column means and which file it comes from |
| `final_evaluation_by_category.csv` | Metrics 1–5 per case category, pooled across all setups and per architecture dimension |

In [7]:
summary.to_csv(DIRECTORY / "final_evaluation_summary.csv", index=False)
guide.to_csv(DIRECTORY / "final_evaluation_columns.csv", index=False)
category_summary.to_csv(DIRECTORY / "final_evaluation_by_category.csv", index=False)
print("Saved the summary, column guide and category breakdown.")

Saved the summary, column guide and category breakdown.
